In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [2]:
df=pd.read_csv("us-shein-mens_clothes-1891.csv")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1890 entries, 0 to 1889
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   color-count          1141 non-null   float64
 1   goods-title-link     1889 non-null   object 
 2   selling_proposition  1277 non-null   object 
 3   price                1889 non-null   object 
 4   discount             1860 non-null   object 
 5   rank-title           560 non-null    object 
 6   rank-sub             560 non-null    object 
dtypes: float64(1), object(6)
memory usage: 103.5+ KB


In [4]:
df.head()

,color-count,goods-title-link,selling_proposition,price,discount,rank-title,rank-sub
0,15.0,Manfinity Homme Loose Fit Men's Solid Color Bu...,10k+ sold recently,$11.99,-7%,NaN,NaN
1,NaN,Manfinity Joysei Men's Star Patterned Shorts W...,NaN,$11.19,-13%,#8 Best Seller,in Blue Men Clothing
2,NaN,Men's Summer Casual Short Sleeve T-Shirt With ...,NaN,$8.09,-13%,#10 Best Seller,in Figure Men T-Shirts
3,11.0,Manfinity Homme Men's Contrast Color Short Sle...,10k+ sold recently,$9.99,-12%,NaN,NaN
4,7.0,Manfinity EMRG Loose Fit Men Plus Letter Print...,30+ sold recently,$3.48,-70%,NaN,NaN


In [5]:
df.tail()

,color-count,goods-title-link,selling_proposition,price,discount,rank-title,rank-sub
1885,NaN,Manfinity Men's Vacation Zebra-Print Drawstrin...,NaN,$11.39,-7%,#9 Best Seller,in Graphic Men Swimwear
1886,NaN,GAWX Men's Casual Color-Block Slogan Printed T...,10+ sold recently,$13.09,-14%,NaN,NaN
1887,NaN,Manfinity Hypemode Men Pocket Front Drawstring...,NaN,$14.29,-23%,#8 Best Seller,in Drawstring Men Shirts
1888,NaN,DAZY Men Patched Pocket Denim Shirt,NaN,$25.39,-7%,#6 Best Seller,in Long Sleeve Men Denim Shirts
1889,4.0,Manfinity Hypemode Men's Loose Fit Pleated Fol...,600+ sold recently,$20.19,-7%,NaN,NaN


In [6]:
# #Errors/Assumptions
# 1.color count contain many null values
# 2.good-title-link all is good just check duplicates and other info
# 3.selling proposition must be in float like 10k->10000
# 4.price convert to indian price(not necessary) also make org_price 
# 5.make discount +ve or remove % 
# 6.rank_title if na fill with NeW 
# 7.rank_sub nan->simple_menswear

In [7]:
df["color-count"].isnull().sum()

749

In [8]:
df["color-count"]=df["color-count"].fillna(0)

In [9]:
df=df.dropna(subset=["goods-title-link"])

In [10]:
df=df.rename(columns={"goods-title-link":"Product_Title"})

In [11]:
df["selling_proposition"]=df["selling_proposition"].str[:5]

In [12]:
df["selling_proposition"]=df["selling_proposition"].str.replace(r"[+s]","",regex=True)

In [13]:
def convert_k(val):
    str_val=str(val).lower().strip()
    if "k" in str_val:
        return int(float(str_val.replace("k",""))*1000)
    else:
        return int(float(str_val)) if str_val.replace(".","").isdigit() else val
        

In [14]:
df["selling_proposition"]=df["selling_proposition"].apply(convert_k)

In [15]:
df["selling_proposition"]=df["selling_proposition"].fillna(0)

In [16]:
#to convert $->₹ 1$=95₹
df["price"]=df["price"].str[1:]

In [17]:
df["price"]=df["price"].astype(float)

In [18]:
df["price"]=df["price"]*95

In [19]:
df=df.rename(columns={"price":"Final_Amount"})


In [20]:
df["Org_Amount"]=""

In [21]:
df["discount"]=df["discount"].str.replace(r"[-%]","",regex=True)

In [22]:
df["discount"]=df["discount"].fillna("0")

In [23]:
df["discount"]=df["discount"].astype(float)

In [24]:
df["Final_Amount"]

0       1139.05
1       1063.05
2        768.55
3        949.05
4        330.60
         ...   
1885    1082.05
1886    1243.55
1887    1357.55
1888    2412.05
1889    1918.05
Name: Final_Amount, Length: 1889, dtype: float64

In [25]:
sp=1139.05
d=7
cp=float()
cp=(sp*100)/(100-d)
print(cp)

1224.784946236559


In [26]:
selling_price=df["Final_Amount"]
discount=df["discount"]
df["Org_Amount"]=(selling_price*100) / (100-discount)

In [27]:
df.head()

,color-count,Product_Title,selling_proposition,Final_Amount,discount,rank-title,rank-sub,Org_Amount
0,15.0,Manfinity Homme Loose Fit Men's Solid Color Bu...,10000.0,1139.05,7.0,NaN,NaN,1224.784946
1,0.0,Manfinity Joysei Men's Star Patterned Shorts W...,0.0,1063.05,13.0,#8 Best Seller,in Blue Men Clothing,1221.896552
2,0.0,Men's Summer Casual Short Sleeve T-Shirt With ...,0.0,768.55,13.0,#10 Best Seller,in Figure Men T-Shirts,883.390805
3,11.0,Manfinity Homme Men's Contrast Color Short Sle...,10000.0,949.05,12.0,NaN,NaN,1078.465909
4,7.0,Manfinity EMRG Loose Fit Men Plus Letter Print...,30.0,330.60,70.0,NaN,NaN,1102.000000


In [28]:
df["rank-title"].isnull().sum()

1329

In [29]:
df["rank-title"]=df["rank-title"].fillna("Unranked(New)")

In [30]:
df["rank-sub"]=df["rank-sub"].fillna("Simple Menswear")

In [31]:
df=df.reset_index(drop=True)

In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1889 entries, 0 to 1888
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   color-count          1889 non-null   float64
 1   Product_Title        1889 non-null   object 
 2   selling_proposition  1889 non-null   float64
 3   Final_Amount         1889 non-null   float64
 4   discount             1889 non-null   float64
 5   rank-title           1889 non-null   object 
 6   rank-sub             1889 non-null   object 
 7   Org_Amount           1889 non-null   float64
dtypes: float64(5), object(3)
memory usage: 118.2+ KB


In [33]:
df.to_csv("D:\Github\python-data-preprocessing\E-Commerce_Sale\E-Commerce-mens_Clothes/cleaned_us-shein-mens_clothes.csv")